# 10 — Opponent-Adjusted Form Features

**Problem:** Current form features (win_rate_5, avg_scored_5, etc.) treat all opponents equally. Beating Moldova 11-1 gives the same form boost as beating Italy 4-1. This inflates teams like Norway who farm weak opponents in Nations League.

**Experiment:** Add 3 new feature groups using existing data:
1. **Strength of Schedule** — `avg_opp_elo_5/10` (average opponent ELO in last 5/10 matches)
2. **ELO-Weighted Win Rate** — wins weighted by opponent ELO (beating strong team counts more)
3. **Performance vs Expected** — `actual_win_rate - elo_expected_win_rate` (overperformance signal)

**Baseline:** Log Loss 0.7988 (XGB×4 + RF×1 + DC×1 blend)

**Input:** `data/processed/matches_clean.csv`, `data/processed/final_elos.csv`
**Output:** If improved → updated model + feature matrix

In [1]:
import pandas as pd
import numpy as np
import joblib
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from scipy.stats import poisson
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, log_loss, classification_report
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style="darkgrid")
PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")

print("Libraries loaded.")

Libraries loaded.


## Step 1 — Load Data & Compute ELO (same as notebook 03)

In [2]:
df = pd.read_csv(PROCESSED_DIR / "matches_clean.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)
print(f"Loaded: {len(df):,} matches ({df['date'].min().date()} to {df['date'].max().date()})")

# ELO computation (same as notebook 03)
def goal_diff_multiplier(gd):
    gd = abs(gd)
    if gd <= 1: return 1.0
    elif gd == 2: return 1.5
    elif gd == 3: return 1.75
    else: return 1.75 + (gd - 3) / 8

def compute_elo(df, initial_rating=1500, home_advantage=100):
    elos = defaultdict(lambda: initial_rating)
    home_elos, away_elos = [], []
    
    for _, row in df.iterrows():
        home, away = row['home_team'], row['away_team']
        home_elo, away_elo = elos[home], elos[away]
        home_elos.append(home_elo)
        away_elos.append(away_elo)
        
        ha = 0 if row['neutral'] else home_advantage
        dr = home_elo - away_elo + ha
        We_home = 1 / (1 + 10 ** (-dr / 400))
        
        gd = row['home_score'] - row['away_score']
        W_home = 1.0 if gd > 0 else (0.5 if gd == 0 else 0.0)
        
        gdm = goal_diff_multiplier(gd)
        K = row['elo_k']
        elos[home] += K * gdm * (W_home - We_home)
        elos[away] += K * gdm * ((1 - W_home) - (1 - We_home))
    
    df = df.copy()
    df['home_elo_before'] = home_elos
    df['away_elo_before'] = away_elos
    df['elo_diff'] = df['home_elo_before'] - df['away_elo_before']
    return df, dict(elos)

print("Computing ELO...")
df, final_elos = compute_elo(df)
print(f"ELO computed for {len(final_elos)} teams.")

Loaded: 60,692 matches (1872-11-30 to 2026-03-31)
Computing ELO...
ELO computed for 342 teams.


## Step 2 — Compute ALL Form Features (Original + New Opponent-Adjusted)

We extend the original `compute_form_features` to also track opponent ELO per match, then compute:
- **Original features** (unchanged): win_rate, avg_scored, avg_conceded, pts_per_match, matches_played
- **New: avg_opp_elo** — mean opponent ELO in last N matches (strength of schedule)
- **New: elo_weighted_win_rate** — wins weighted by `opp_elo / 1500` (beating ELO 2000 team counts 1.33x)
- **New: performance_vs_expected** — actual win rate minus ELO-expected win rate (true overperformance)

In [3]:
def compute_form_features_v2(df, windows=[5, 10], exclude_friendly=True):
    """
    Extended form features — original + opponent-adjusted.
    
    For each match, computes features for both home and away team
    using their last N competitive matches BEFORE the current match.
    
    New features per window per team:
      - avg_opp_elo_N: mean opponent ELO (strength of schedule)
      - elo_weighted_win_rate_N: wins weighted by opp_elo/1500
      - performance_vs_expected_N: actual_win_rate - elo_expected_win_rate
    """
    team_history = defaultdict(list)
    results = []
    
    for idx, row in df.iterrows():
        home, away = row['home_team'], row['away_team']
        is_friendly = row['tournament_category'] == 'friendly'
        
        row_features = {}
        
        for team, prefix, goals_scored, goals_conceded, opp_elo_before in [
            (home, 'home', row['home_score'], row['away_score'], row['away_elo_before']),
            (away, 'away', row['away_score'], row['home_score'], row['home_elo_before']),
        ]:
            history = team_history[team]
            comp_history = [h for h in history if not h['friendly']] if exclude_friendly else history
            
            for N in windows:
                recent = comp_history[-N:] if len(comp_history) >= 1 else []
                n = len(recent)
                
                if n == 0:
                    # Neutral priors
                    win_rate = 0.5
                    avg_scored = 1.0
                    avg_conceded = 1.0
                    pts_per_match = 1.0
                    avg_opp_elo = 1500.0
                    elo_weighted_wr = 0.5
                    perf_vs_expected = 0.0
                else:
                    wins = sum(1 for h in recent if h['result'] == 'W')
                    draws = sum(1 for h in recent if h['result'] == 'D')
                    win_rate = wins / n
                    avg_scored = sum(h['scored'] for h in recent) / n
                    avg_conceded = sum(h['conceded'] for h in recent) / n
                    pts_per_match = (wins * 3 + draws * 1) / n
                    
                    # NEW: Strength of schedule
                    avg_opp_elo = sum(h['opp_elo'] for h in recent) / n
                    
                    # NEW: ELO-weighted win rate
                    # Weight each win by opp_elo/1500 — beating a 2000 ELO team counts 1.33x
                    weighted_wins = sum(
                        (h['opp_elo'] / 1500) for h in recent if h['result'] == 'W'
                    )
                    total_weight = sum(h['opp_elo'] / 1500 for h in recent)
                    elo_weighted_wr = weighted_wins / total_weight if total_weight > 0 else 0.5
                    
                    # NEW: Performance vs ELO-expected
                    # For each match, expected win prob from ELO: 1/(1+10^(-elo_diff/400))
                    expected_wins = sum(h['expected_win_prob'] for h in recent)
                    expected_wr = expected_wins / n
                    perf_vs_expected = win_rate - expected_wr
                
                # Original features
                row_features[f'{prefix}_win_rate_{N}'] = win_rate
                row_features[f'{prefix}_avg_scored_{N}'] = avg_scored
                row_features[f'{prefix}_avg_conceded_{N}'] = avg_conceded
                row_features[f'{prefix}_pts_per_match_{N}'] = pts_per_match
                row_features[f'{prefix}_matches_played_{N}'] = n
                
                # New features
                row_features[f'{prefix}_avg_opp_elo_{N}'] = avg_opp_elo
                row_features[f'{prefix}_elo_weighted_wr_{N}'] = elo_weighted_wr
                row_features[f'{prefix}_perf_vs_expected_{N}'] = perf_vs_expected
        
        results.append(row_features)
        
        # Update history AFTER recording (no leakage)
        home_result = 'W' if row['home_score'] > row['away_score'] else ('D' if row['home_score'] == row['away_score'] else 'L')
        away_result = 'W' if row['away_score'] > row['home_score'] else ('D' if row['home_score'] == row['away_score'] else 'L')
        
        # Expected win prob from ELO (for performance_vs_expected)
        home_elo_diff = row['home_elo_before'] - row['away_elo_before']
        home_expected = 1 / (1 + 10 ** (-home_elo_diff / 400))
        
        team_history[home].append({
            'scored': row['home_score'], 'conceded': row['away_score'],
            'result': home_result, 'friendly': is_friendly,
            'opp_elo': row['away_elo_before'],
            'expected_win_prob': home_expected,
        })
        team_history[away].append({
            'scored': row['away_score'], 'conceded': row['home_score'],
            'result': away_result, 'friendly': is_friendly,
            'opp_elo': row['home_elo_before'],
            'expected_win_prob': 1 - home_expected,
        })
    
    return pd.DataFrame(results, index=df.index)

print("Extended form function defined. Computing...")
form_df = compute_form_features_v2(df)
print(f"Done. Shape: {form_df.shape}")
print(f"\nNew columns added:")
new_cols = [c for c in form_df.columns if 'opp_elo' in c or 'weighted_wr' in c or 'perf_vs' in c]
for c in new_cols:
    print(f"  {c}")

Extended form function defined. Computing...
Done. Shape: (60692, 32)

New columns added:
  home_avg_opp_elo_5
  home_elo_weighted_wr_5
  home_perf_vs_expected_5
  home_avg_opp_elo_10
  home_elo_weighted_wr_10
  home_perf_vs_expected_10
  away_avg_opp_elo_5
  away_elo_weighted_wr_5
  away_perf_vs_expected_5
  away_avg_opp_elo_10
  away_elo_weighted_wr_10
  away_perf_vs_expected_10


## Step 3 — Sanity Check: Norway vs Spain

Let's verify the new features capture what we expect — Norway should have low avg_opp_elo (weak opponents) and high perf_vs_expected should be near 0 (winning matches they were expected to win).

In [4]:
# Get latest form for key teams
teams_check = ['Spain', 'France', 'Argentina', 'England', 'Norway', 'Mexico', 'Brazil', 'Switzerland']

latest_home = form_df.loc[df.groupby('home_team').apply(lambda g: g.index[-1])]
latest_home['team'] = df.loc[latest_home.index, 'home_team'].values

print(f"{'Team':15s} {'win5':>6s} {'opp_elo5':>9s} {'elo_wr5':>8s} {'perf5':>7s} | {'win10':>6s} {'opp_elo10':>10s} {'elo_wr10':>9s} {'perf10':>7s}")
print("-" * 95)
for t in teams_check:
    row = latest_home[latest_home['team'] == t]
    if len(row) == 0:
        continue
    r = row.iloc[0]
    print(f"{t:15s} {r['home_win_rate_5']:6.3f} {r['home_avg_opp_elo_5']:9.0f} {r['home_elo_weighted_wr_5']:8.3f} {r['home_perf_vs_expected_5']:+7.3f} | "
          f"{r['home_win_rate_10']:6.3f} {r['home_avg_opp_elo_10']:10.0f} {r['home_elo_weighted_wr_10']:9.3f} {r['home_perf_vs_expected_10']:+7.3f}")

Team              win5  opp_elo5  elo_wr5   perf5 |  win10  opp_elo10  elo_wr10  perf10
-----------------------------------------------------------------------------------------------
Spain            0.800      1805    0.781  -0.117 |  0.600       1885     0.567  -0.254
France           0.800      1617    0.796  -0.142 |  0.800       1818     0.785  +0.007
Argentina        0.600      1981    0.589  -0.205 |  0.600       1933     0.594  -0.237
England          1.000      1625    1.000  +0.111 |  1.000       1526     1.000  +0.074
Norway           1.000      1603    1.000  +0.166 |  1.000       1635     1.000  +0.198
Mexico           0.600      1800    0.596  -0.155 |  0.800       1744     0.792  +0.013
Brazil           0.400      1897    0.395  -0.353 |  0.400       1965     0.392  -0.275
Switzerland      0.600      1822    0.599  -0.139 |  0.400       1885     0.381  -0.223


## Step 4 — Assemble Full Feature Matrix (Original + H2H + Static + DC + New)

We rebuild the complete feature matrix with the new columns, keeping everything else identical to notebook 03/06.

In [5]:
# H2H features (same as notebook 03)
def compute_h2h_features(df):
    h2h_history = defaultdict(list)
    results = []
    for _, row in df.iterrows():
        home, away = row['home_team'], row['away_team']
        h2h = h2h_history[(home, away)] + [
            {'scored': h['conceded'], 'conceded': h['scored'], 'result': {'W':'L','L':'W','D':'D'}[h['result']]}
            for h in h2h_history[(away, home)]
        ]
        n = len(h2h)
        recent_h2h = h2h[-5:]
        n_recent = len(recent_h2h)
        
        if n == 0:
            h2h_hw, h2h_hs, h2h_hc = 0.5, 1.0, 1.0
        else:
            h2h_hw = sum(1 for h in h2h if h['result'] == 'W') / n
            h2h_hs = sum(h['scored'] for h in h2h) / n
            h2h_hc = sum(h['conceded'] for h in h2h) / n
        
        h2h_rw = (sum(1 for h in recent_h2h if h['result'] == 'W') / n_recent) if n_recent > 0 else 0.5
        
        results.append({
            'h2h_home_win_rate': h2h_hw, 'h2h_home_avg_scored': h2h_hs,
            'h2h_home_avg_conceded': h2h_hc, 'h2h_total_meetings': n,
            'h2h_recent_win_rate': h2h_rw,
        })
        
        home_result = 'W' if row['home_score'] > row['away_score'] else ('D' if row['home_score'] == row['away_score'] else 'L')
        h2h_history[(home, away)].append({
            'scored': row['home_score'], 'conceded': row['away_score'], 'result': home_result
        })
    return pd.DataFrame(results, index=df.index)

print("Computing H2H features...")
h2h_df = compute_h2h_features(df)
print(f"H2H shape: {h2h_df.shape}")

# Static features (same as notebook 03)
CONFEDERATIONS = ['UEFA', 'CAF', 'AFC', 'CONCACAF', 'CONMEBOL', 'OFC', 'UNKNOWN']
TOURNAMENT_ORDINAL = {'friendly': 0, 'other_competitive': 1, 'qualifier': 2, 'continental_final': 3, 'world_cup': 4}

static_features = pd.DataFrame(index=df.index)
static_features['neutral'] = df['neutral'].astype(int)
static_features['tournament_importance'] = df['tournament_category'].map(TOURNAMENT_ORDINAL)
for conf in CONFEDERATIONS:
    static_features[f'home_conf_{conf}'] = (df['home_confederation'] == conf).astype(int)
    static_features[f'away_conf_{conf}'] = (df['away_confederation'] == conf).astype(int)
static_features['same_confederation'] = (df['home_confederation'] == df['away_confederation']).astype(int)
print(f"Static features shape: {static_features.shape}")

# Assemble
META_COLS = ['date', 'home_team', 'away_team', 'home_score', 'away_score',
             'tournament', 'tournament_category', 'neutral', 'outcome', 'goal_diff']

feature_matrix = pd.concat([
    df[META_COLS].reset_index(drop=True),
    df[['home_elo_before', 'away_elo_before', 'elo_diff']].reset_index(drop=True),
    form_df.reset_index(drop=True),
    h2h_df.reset_index(drop=True),
    static_features.reset_index(drop=True),
], axis=1)

# Add engineered features (same as notebook 06 autoresearch)
feature_matrix['elo_diff_sq'] = feature_matrix['elo_diff']**2 * np.sign(feature_matrix['elo_diff'])
feature_matrix['home_form_momentum'] = feature_matrix['home_win_rate_5'] - feature_matrix['home_win_rate_10']
feature_matrix['away_form_momentum'] = feature_matrix['away_win_rate_5'] - feature_matrix['away_win_rate_10']
feature_matrix['home_goal_diff_form'] = feature_matrix['home_avg_scored_5'] - feature_matrix['home_avg_conceded_5']
feature_matrix['away_goal_diff_form'] = feature_matrix['away_avg_scored_5'] - feature_matrix['away_avg_conceded_5']
feature_matrix['net_goal_diff'] = feature_matrix['home_goal_diff_form'] - feature_matrix['away_goal_diff_form']
feature_matrix['h2h_confidence'] = feature_matrix['h2h_recent_win_rate'] * (feature_matrix['h2h_total_meetings'] / (feature_matrix['h2h_total_meetings'] + 5))

# NEW: Derived opponent-adjusted features
feature_matrix['sos_diff_5'] = feature_matrix['home_avg_opp_elo_5'] - feature_matrix['away_avg_opp_elo_5']
feature_matrix['sos_diff_10'] = feature_matrix['home_avg_opp_elo_10'] - feature_matrix['away_avg_opp_elo_10']

# Encode outcome
outcome_map = {'home_win': 1, 'draw': 0, 'away_win': -1}
feature_matrix['outcome_num'] = feature_matrix['outcome'].map(outcome_map)

print(f"\nFull feature matrix: {feature_matrix.shape}")
print(f"Total columns: {feature_matrix.shape[1]}")

Computing H2H features...
H2H shape: (60692, 5)
Static features shape: (60692, 17)

Full feature matrix: (60692, 77)
Total columns: 77


## Step 5 — Correlation Check: Do New Features Add Signal?

Check if the new features correlate with match outcome independently of ELO.

In [6]:
# Correlation of new features with outcome
new_feature_cols = [c for c in feature_matrix.columns if 'opp_elo' in c or 'weighted_wr' in c or 'perf_vs' in c or 'sos_diff' in c]
old_key_features = ['elo_diff', 'home_win_rate_5', 'away_win_rate_5', 'h2h_home_win_rate']

all_check = old_key_features + new_feature_cols
correlations = feature_matrix[all_check + ['outcome_num']].corr()['outcome_num'].drop('outcome_num').sort_values(ascending=False)

print("Feature correlation with outcome (new features marked with *):")
print("=" * 65)
for feat, corr in correlations.items():
    marker = " *" if feat in new_feature_cols else "  "
    bar = '█' * int(abs(corr) * 50)
    sign = '+' if corr >= 0 else '-'
    print(f"{marker} {feat:40s} {sign}{abs(corr):.4f} {bar}")

Feature correlation with outcome (new features marked with *):
   elo_diff                                 +0.4764 ███████████████████████
   h2h_home_win_rate                        +0.3435 █████████████████
 * home_elo_weighted_wr_10                  +0.1935 █████████
 * sos_diff_10                              +0.1835 █████████
   home_win_rate_5                          +0.1784 ████████
 * home_elo_weighted_wr_5                   +0.1783 ████████
 * sos_diff_5                               +0.1220 ██████
 * home_avg_opp_elo_10                      +0.0465 ██
 * home_avg_opp_elo_5                       +0.0251 █
 * home_perf_vs_expected_5                  +0.0184 
 * home_perf_vs_expected_10                 +0.0087 
 * away_perf_vs_expected_10                 -0.0090 
 * away_perf_vs_expected_5                  -0.0134 
 * away_avg_opp_elo_5                       -0.0854 ████
 * away_avg_opp_elo_10                      -0.1121 █████
 * away_elo_weighted_wr_5                   -0.175

## Step 6 — Train & Compare: Baseline (59 features) vs Extended (59 + 14 new)

We train XGB + RF with exact same hyperparameters as the best model, then add DC as voter.

**Comparison A:** Original 59 features (reproduce 0.7988 baseline)
**Comparison B:** 59 + 14 new opponent-adjusted features

In [7]:
# Load DC model for DC-as-voter features
dc_bundle = joblib.load(MODELS_DIR / 'dc_model.pkl')
attack_params = dc_bundle['attack']
defense_params = dc_bundle['defense']
team_idx_dc = dc_bundle['team_idx']
with open(MODELS_DIR / 'dc_params.json') as f:
    dc_params = json.load(f)
home_adv_dc = dc_params['home_adv']
rho_dc = dc_params['rho']
MAX_GOALS = 10

def dc_match_probs(home, away, neutral=True):
    if home not in team_idx_dc or away not in team_idx_dc:
        return 0.45, 0.25, 0.30
    goals = np.arange(MAX_GOALS + 1)
    hi, ai = team_idx_dc[home], team_idx_dc[away]
    ha = home_adv_dc if not neutral else 0.0
    lam = np.exp(attack_params[hi] + defense_params[ai] + ha)
    mu = np.exp(attack_params[ai] + defense_params[hi])
    score_mat = np.outer(poisson.pmf(goals, lam), poisson.pmf(goals, mu))
    score_mat[0, 0] *= max(1 - lam * mu * rho_dc, 1e-10)
    score_mat[0, 1] *= max(1 + lam * rho_dc, 1e-10)
    score_mat[1, 0] *= max(1 + mu * rho_dc, 1e-10)
    score_mat[1, 1] *= max(1 - rho_dc, 1e-10)
    home_mask = goals[:, None] > goals[None, :]
    draw_mask = goals[:, None] == goals[None, :]
    away_mask = goals[:, None] < goals[None, :]
    hw = np.sum(score_mat * home_mask)
    dr = np.sum(score_mat * draw_mask)
    aw = np.sum(score_mat * away_mask)
    total = hw + dr + aw
    return hw/total, dr/total, aw/total

# Fix: deduplicate 'neutral' column (appears in both META_COLS and static_features)
feature_matrix = feature_matrix.loc[:, ~feature_matrix.columns.duplicated(keep='last')]
print(f"After dedup: {feature_matrix.shape} columns")

# Add DC features to feature matrix
print("Computing DC features for all matches...")
dc_features = []
for _, row in feature_matrix.iterrows():
    h, a = row['home_team'], row['away_team']
    is_neutral = bool(row['neutral'])
    dc_hw, dc_dr, dc_aw = dc_match_probs(h, a, is_neutral)
    hi = team_idx_dc.get(h, 0)
    ai = team_idx_dc.get(a, 0)
    ha = home_adv_dc if not is_neutral else 0.0
    dc_lam = np.exp(attack_params[hi] + defense_params[ai] + ha)
    dc_mu = np.exp(attack_params[ai] + defense_params[hi])
    dc_features.append({
        'dc_home_win_prob': dc_hw, 'dc_draw_prob': dc_dr, 'dc_away_win_prob': dc_aw,
        'dc_lambda': dc_lam, 'dc_mu': dc_mu,
        'dc_total_goals': dc_lam + dc_mu, 'dc_goal_diff': dc_lam - dc_mu,
    })
dc_df = pd.DataFrame(dc_features)
for col in dc_df.columns:
    feature_matrix[col] = dc_df[col].values
print(f"DC features added. Matrix shape: {feature_matrix.shape}")

After dedup: (60692, 76) columns
Computing DC features for all matches...
DC features added. Matrix shape: (60692, 83)


In [8]:
# Define feature sets
EXCLUDE = ['date', 'home_team', 'away_team', 'home_score', 'away_score',
           'tournament', 'tournament_category', 'neutral', 'outcome', 'goal_diff',
           'outcome_num']

# Original 59 features (same as notebook 07, but 'neutral.1' → 'neutral' after dedup)
ORIGINAL_FEATURES = [
    'home_elo_before', 'away_elo_before', 'elo_diff',
    'home_win_rate_5', 'home_avg_scored_5', 'home_avg_conceded_5', 'home_pts_per_match_5', 'home_matches_played_5',
    'home_win_rate_10', 'home_avg_scored_10', 'home_avg_conceded_10', 'home_pts_per_match_10', 'home_matches_played_10',
    'away_win_rate_5', 'away_avg_scored_5', 'away_avg_conceded_5', 'away_pts_per_match_5', 'away_matches_played_5',
    'away_win_rate_10', 'away_avg_scored_10', 'away_avg_conceded_10', 'away_pts_per_match_10', 'away_matches_played_10',
    'h2h_home_win_rate', 'h2h_home_avg_scored', 'h2h_home_avg_conceded', 'h2h_total_meetings', 'h2h_recent_win_rate',
    'neutral', 'tournament_importance',
    'home_conf_UEFA', 'home_conf_CAF', 'home_conf_AFC', 'home_conf_CONCACAF', 'home_conf_CONMEBOL', 'home_conf_OFC', 'home_conf_UNKNOWN',
    'away_conf_UEFA', 'away_conf_CAF', 'away_conf_AFC', 'away_conf_CONCACAF', 'away_conf_CONMEBOL', 'away_conf_OFC', 'away_conf_UNKNOWN',
    'same_confederation',
    'elo_diff_sq', 'home_form_momentum', 'away_form_momentum',
    'home_goal_diff_form', 'away_goal_diff_form', 'net_goal_diff', 'h2h_confidence',
    'dc_home_win_prob', 'dc_draw_prob', 'dc_away_win_prob',
    'dc_lambda', 'dc_mu', 'dc_total_goals', 'dc_goal_diff',
]

# New features to add
NEW_FEATURES = [
    'home_avg_opp_elo_5', 'home_avg_opp_elo_10',
    'away_avg_opp_elo_5', 'away_avg_opp_elo_10',
    'home_elo_weighted_wr_5', 'home_elo_weighted_wr_10',
    'away_elo_weighted_wr_5', 'away_elo_weighted_wr_10',
    'home_perf_vs_expected_5', 'home_perf_vs_expected_10',
    'away_perf_vs_expected_5', 'away_perf_vs_expected_10',
    'sos_diff_5', 'sos_diff_10',
]

EXTENDED_FEATURES = ORIGINAL_FEATURES + NEW_FEATURES

print(f"Original features: {len(ORIGINAL_FEATURES)}")
print(f"New features: {len(NEW_FEATURES)}")
print(f"Extended features: {len(EXTENDED_FEATURES)}")

# Verify all features exist
missing = [c for c in EXTENDED_FEATURES if c not in feature_matrix.columns]
if missing:
    print(f"\nMISSING COLUMNS: {missing}")
else:
    print("\nAll features present in matrix. ✓")

Original features: 59
New features: 14
Extended features: 73

All features present in matrix. ✓


In [9]:
# Train/test split (same date as original)
SPLIT_DATE = '2022-11-20'
model_data = feature_matrix[feature_matrix['tournament_category'] != 'friendly'].copy()
train = model_data[model_data['date'] < SPLIT_DATE]
test = model_data[model_data['date'] >= SPLIT_DATE]
print(f"Train: {len(train):,} | Test: {len(test):,}")

le = LabelEncoder()
le.fit(['away_win', 'draw', 'home_win'])
y_train = le.transform(train['outcome'])
y_test = le.transform(test['outcome'])

# Class weights — matches train.py exactly
from sklearn.utils.class_weight import compute_class_weight
from sklearn.calibration import CalibratedClassifierCV

classes = np.unique(y_train)  # [0, 1, 2] — integer encoded
weights = compute_class_weight('balanced', classes=classes, y=y_train)
cw = dict(zip(classes, weights))  # {0: w, 1: w, 2: w} — integer keys!

def train_and_evaluate(feature_cols, label):
    """Train XGB+RF (isotonic calibrated), blend with DC×5, return log loss.
    Matches train.py exactly."""
    X_tr = train[feature_cols].values
    X_te = test[feature_cols].values
    
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    
    # XGBoost + isotonic calibration (same as train.py)
    xgb_base = xgb.XGBClassifier(
        n_estimators=500, max_depth=5, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
        eval_metric='mlogloss', objective='multi:softprob',
        random_state=42, n_jobs=-1, verbosity=0
    )
    xgb_model = CalibratedClassifierCV(xgb_base, method='isotonic', cv=5)
    xgb_model.fit(X_tr_s, y_train)
    
    # RF + isotonic calibration (same as train.py)
    rf_base = RandomForestClassifier(
        n_estimators=300, max_depth=12, min_samples_leaf=5,
        class_weight=cw, random_state=42, n_jobs=-1
    )
    rf_model = CalibratedClassifierCV(rf_base, method='isotonic', cv=5)
    rf_model.fit(X_tr_s, y_train)
    
    # Predict
    xgb_p = xgb_model.predict_proba(X_te_s)
    rf_p = rf_model.predict_proba(X_te_s)
    
    # DC as voter — use ACTUAL neutral flag per match
    dc_probs = []
    for _, row in test.iterrows():
        is_neutral = bool(row['neutral'])
        dc_hw, dc_dr, dc_aw = dc_match_probs(row['home_team'], row['away_team'], is_neutral)
        dc_probs.append([dc_aw, dc_dr, dc_hw])
    dc_p = np.array(dc_probs)
    dc_p = dc_p / dc_p.sum(axis=1, keepdims=True)
    
    # Blend: XGB×4 + RF×1 + DC×5 (same as train.py)
    W_XGB, W_RF, W_DC = 4, 1, 5
    blended = (W_XGB * xgb_p + W_RF * rf_p + W_DC * dc_p) / (W_XGB + W_RF + W_DC)
    
    ll = log_loss(y_test, blended)
    
    xgb_ll = log_loss(y_test, xgb_p)
    blend_no_dc = (W_XGB * xgb_p + W_RF * rf_p) / (W_XGB + W_RF)
    no_dc_ll = log_loss(y_test, blend_no_dc)
    
    y_pred = np.argmax(blended, axis=1)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    
    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")
    print(f"  Features:       {len(feature_cols)}")
    print(f"  XGB-only LL:    {xgb_ll:.4f}")
    print(f"  XGB+RF LL:      {no_dc_ll:.4f}")
    print(f"  XGB+RF+DC LL:   {ll:.4f}  {'← should be ~0.7988' if len(feature_cols) == 59 else ''}")
    print(f"  Accuracy:       {acc:.4f}")
    print(f"  F1 Macro:       {f1:.4f}")
    
    return {
        'label': label, 'features': len(feature_cols),
        'xgb_ll': xgb_ll, 'no_dc_ll': no_dc_ll, 'blended_ll': ll,
        'accuracy': acc, 'f1': f1,
        'xgb_model': xgb_model, 'rf_model': rf_model, 'scaler': scaler,
    }

print("Training function defined.")

Train: 35,304 | Test: 3,313
Training function defined.


In [10]:
# Run both experiments
print("EXPERIMENT A: Baseline (59 features)")
result_a = train_and_evaluate(ORIGINAL_FEATURES, "Baseline (59 features)")

print("\n\nEXPERIMENT B: Extended (59 + 14 opponent-adjusted)")
result_b = train_and_evaluate(EXTENDED_FEATURES, "Extended (73 features)")

# Summary
print("\n\n" + "=" * 60)
print("COMPARISON SUMMARY")
print("=" * 60)
print(f"{'':30s} {'Baseline':>10s} {'Extended':>10s} {'Delta':>10s}")
print("-" * 60)
print(f"{'Features':30s} {result_a['features']:>10d} {result_b['features']:>10d}")
print(f"{'XGB-only Log Loss':30s} {result_a['xgb_ll']:>10.4f} {result_b['xgb_ll']:>10.4f} {result_b['xgb_ll'] - result_a['xgb_ll']:>+10.4f}")
print(f"{'XGB+RF Log Loss':30s} {result_a['no_dc_ll']:>10.4f} {result_b['no_dc_ll']:>10.4f} {result_b['no_dc_ll'] - result_a['no_dc_ll']:>+10.4f}")
print(f"{'XGB+RF+DC Log Loss':30s} {result_a['blended_ll']:>10.4f} {result_b['blended_ll']:>10.4f} {result_b['blended_ll'] - result_a['blended_ll']:>+10.4f}")
print(f"{'Accuracy':30s} {result_a['accuracy']:>10.4f} {result_b['accuracy']:>10.4f} {result_b['accuracy'] - result_a['accuracy']:>+10.4f}")
print(f"{'F1 Macro':30s} {result_a['f1']:>10.4f} {result_b['f1']:>10.4f} {result_b['f1'] - result_a['f1']:>+10.4f}")

improved = result_b['blended_ll'] < result_a['blended_ll']
delta = result_a['blended_ll'] - result_b['blended_ll']
print(f"\n{'✅ IMPROVED' if improved else '❌ NO IMPROVEMENT'} — Delta: {delta:+.4f}")

EXPERIMENT A: Baseline (59 features)

  Baseline (59 features)
  Features:       59
  XGB-only LL:    0.8189
  XGB+RF LL:      0.8177
  XGB+RF+DC LL:   0.7987  ← should be ~0.7988
  Accuracy:       0.6399
  F1 Macro:       0.4962


EXPERIMENT B: Extended (59 + 14 opponent-adjusted)

  Extended (73 features)
  Features:       73
  XGB-only LL:    0.8146
  XGB+RF LL:      0.8135
  XGB+RF+DC LL:   0.7967  
  Accuracy:       0.6387
  F1 Macro:       0.4958


COMPARISON SUMMARY
                                 Baseline   Extended      Delta
------------------------------------------------------------
Features                               59         73
XGB-only Log Loss                  0.8189     0.8146    -0.0043
XGB+RF Log Loss                    0.8177     0.8135    -0.0041
XGB+RF+DC Log Loss                 0.7987     0.7967    -0.0020
Accuracy                           0.6399     0.6387    -0.0012
F1 Macro                           0.4962     0.4958    -0.0004

✅ IMPROVED — Delta: +0.

## Step 7 — Feature Importance: What Did the New Features Contribute?

In [13]:
# Feature importance from the extended model
# CalibratedClassifierCV wraps the base XGB — dig into the first calibrated classifier
cal_clf = result_b['xgb_model'].calibrated_classifiers_[0]
xgb_inner = getattr(cal_clf, 'estimator', None) or getattr(cal_clf, 'base_estimator', None)
print(f"Inner model type: {type(xgb_inner)}")

importances = pd.Series(
    xgb_inner.feature_importances_,
    index=EXTENDED_FEATURES
).sort_values(ascending=False)

print("\nTop 25 features by XGBoost importance (extended model):")
print("=" * 55)
for i, (feat, imp) in enumerate(importances.head(25).items()):
    marker = " *NEW*" if feat in NEW_FEATURES else ""
    bar = '█' * int(imp * 200)
    print(f"  {i+1:2d}. {feat:35s} {imp:.4f} {bar}{marker}")

print(f"\nNew features in top 25: {sum(1 for f in importances.head(25).index if f in NEW_FEATURES)}/14")
print(f"\nAll new feature importances:")
for feat in NEW_FEATURES:
    imp = importances.get(feat, 0)
    rank = list(importances.index).index(feat) + 1 if feat in importances.index else -1
    print(f"  {feat:35s} importance={imp:.4f}  rank={rank}/{len(EXTENDED_FEATURES)}")

Inner model type: <class 'xgboost.sklearn.XGBClassifier'>

Top 25 features by XGBoost importance (extended model):
   1. elo_diff_sq                         0.1598 ███████████████████████████████
   2. elo_diff                            0.1457 █████████████████████████████
   3. h2h_recent_win_rate                 0.0193 ███
   4. away_conf_CAF                       0.0159 ███
   5. dc_away_win_prob                    0.0155 ███
   6. net_goal_diff                       0.0153 ███
   7. dc_goal_diff                        0.0152 ███
   8. home_conf_CAF                       0.0131 ██
   9. h2h_home_win_rate                   0.0128 ██
  10. away_conf_UEFA                      0.0128 ██
  11. neutral                             0.0125 ██
  12. home_matches_played_10              0.0125 ██
  13. home_conf_UNKNOWN                   0.0124 ██
  14. tournament_importance               0.0119 ██
  15. home_avg_conceded_10                0.0115 ██
  16. home_avg_conceded_5                 0.

## Step 8 — Next Steps

If the extended model improved log loss:
- Save new model files (`best_model.pkl`, `scaler_dc.pkl`, `feature_cols_dc.pkl`)
- Update `features_matrix.csv` with new columns
- Update backend `predictor.py` with new feature computation
- Re-run 10K simulation to see updated championship probabilities

If no improvement:
- Check which new features had zero/low importance (might be redundant with ELO)
- Try ablation: only add `avg_opp_elo` without the other two
- Consider that ELO already captures opponent strength implicitly